# Identificação de ansiedade

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
!pip install Unidecode

In [ ]:
import pandas as pd
import re
import random
import numpy as np
import torch

# Loading data

In [ ]:
url = ''
file_id = url.split('/')[-2]
read_url='https://drive.google.com/uc?id=' + file_id

data_set = pd.read_excel(read_url, index_col=False)

condition = [
    data_set["rotulo_humano"] == "sem_sintoma", # 0
    data_set["rotulo_humano"] == "sintoma" # 1
]

values = [0, 1]

data_set["classification"] = np.select(condition, values)

data_set

# Unsloth

In [ ]:
from unsloth import FastLanguageModel

fourbit_models = [
    "unsloth/Qwen3-1.7B-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

<a name="Inference"></a>
### **Inference**
De acordo com a equipe `Qwen-3`, as configurações recomendadas para inferência de raciocínio são `temperature = 0.6, top_p = 0.95, top_k = 20`

Para inferência baseada em bate-papo normal, `temperature = 0.7, top_p = 0.8, top_k = 20`

In [ ]:
from tqdm import tqdm
import torch

answers_llm = {
    "text": data_set["Texto"].tolist(),
    "target": data_set["classification"].tolist(),
    "predicted": []
}

sentences = answers_llm["text"]

FastLanguageModel.for_inference(model)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

batch_size = 16
for i in tqdm(range(0, len(sentences), batch_size)):
    batch = sentences[i:i+batch_size]

    prompts = [
        f"""
        Você é um modelo de IA treinado para classificar frases quanto à ansiedade e depressão.

        Regras:
        - Se a frase indicar ansiedade ou depressão, retorne exatamente "1".
        - Se a frase não indicar ansiedade ou depressão, retorne exatamente "0".
        - Não adicione explicações.

        Frase: "{sentence}"
        """
        for sentence in batch
    ]

    texts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        for prompt in prompts
    ]

    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            temperature=0.6,
            top_p=0.95,
            top_k=20,
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    preds = []
    for text in decoded:
        if "0" in text:
            preds.append("0")
        elif "1" in text:
            preds.append("1")
        else:
            preds.append("")

    answers_llm["predicted"].extend(preds)


In [ ]:
# from google.colab import files

df = pd.DataFrame(answers_llm)
df.to_csv('/content/drive/MyDrive/Notebooks/Others/Outputs/respostas_qwen3.csv', index=False)

# files.download('respostas_qwen3.xlsx')

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

y_true = answers_llm["target"]
y_pred = answers_llm["predicted"]

y_true = [int(x) for x in y_true]
y_pred = [int(x.strip()) if x.strip() in ['0', '1'] else 0 for x in y_pred]

acc = accuracy_score(y_true, y_pred)
print(f"Acurácia: {acc:.4f}")

print("\nRelatório de Classificação:")
print(classification_report(y_true, y_pred, target_names=["Sem ansiedade (0)", "Com ansiedade (1)"]))


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

def plot_confusion_matrix(y_preds, y_true, labels=None):
  cm = confusion_matrix(y_true, y_preds, normalize="true")
  fig, ax = plt.subplots(figsize=(6, 6))
  disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
  disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
  plt.title("Normalized confusion matrix")
  plt.show()

In [ ]:
plot_confusion_matrix(y_pred, y_true, labels=["Sem ansiedade (0)", "Com ansiedade (1)"])